# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for accessing, exploring, and processing the FAIR² dataset—an ordered logistic regression study of rangeland management practices—using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset adheres to the [MLCommons Croissant](https://mlcommons.org/croissant/) metadata specification and is published as FAIR data via [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets and fields. Each entity is referenced by its Croissant `@id`.

In [ ]:
# List all RecordSets with their @id
print("Record sets available in the dataset (by @id):")
record_sets = metadata.recordSet if hasattr(metadata, "recordSet") and metadata.recordSet else []
if not record_sets:
    print("No record sets defined at top-level. Querying known datasets...")
    # Try infer from dataset object
    if hasattr(dataset, "record_sets"):
        record_sets_dict = dataset.record_sets
        if record_sets_dict:
            for rs_id in record_sets_dict:
                print(f"- {rs_id}")
    else:
        print("No record sets found.")
else:
    for rs in record_sets:
        print(f"- {getattr(rs, '@id', str(rs))}")

# Inspect fields (columns) for each detected record set by @id
def print_fields_for_recordset(rs_id):
    print(f"\nFields for record set '{rs_id}':")
    try:
        # dataset.schema helps navigate the croissant structure
        schema = dataset.schema
        for entity in schema.get('@graph', []):
            if entity.get('@id') == rs_id and 'field' in entity:
                for field in entity['field']:
                    if isinstance(field, dict) and '@id' in field:
                        print(f"  - {field['@id']}")
                    elif isinstance(field, str):
                        print(f"  - {field}")
                return
        print('  (No fields found)')
    except Exception as e:
        print(f'  (Failed to retrieve fields: {e})')

# Enumerate record set IDs from the live dataset object
all_record_set_ids = []
if hasattr(dataset, 'record_sets') and dataset.record_sets:
    all_record_set_ids = list(dataset.record_sets.keys())
elif record_sets:
    all_record_set_ids = [getattr(rs, '@id', str(rs)) for rs in record_sets]
else:
    print("No record sets found.")

for rs_id in all_record_set_ids:
    print_fields_for_recordset(rs_id)


## 3. Data Extraction
Load records from each record set into a pandas DataFrame using their `@id`s for reference.
You should use the printed record set IDs and their field IDs from the overview above.

In [ ]:
# Build DataFrames for each available record set.
import warnings
warnings.filterwarnings('ignore')

# Use the record set IDs found earlier. If empty, document and skip.
if not all_record_set_ids:
    print("No record sets detected in metadata. Unable to extract tabular data.")
else:
    dataframes = {}
    for rs_id in all_record_set_ids:
        print(f"Reading records from record set: {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[rs_id] = df
                print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
            else:
                print("  (No records found)")
        except Exception as e:
            print(f"  (Error reading records: {e})")

    # Preview columns for the first non-empty DataFrame
    for rs_id, df in dataframes.items():
        print(f"\nSample records from '{rs_id}':")
        print(df.head())
        break  # Only preview the first non-empty record set

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering by numeric field, normalization, and group-wise computations. All fields are referenced by their `@id`.

Pick a record set, choose a numeric field (by `@id`), and a group field for analysis.

In [ ]:
# For demonstrative purposes, select the first loaded record set containing numeric columns.

selected_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs_id, df in dataframes.items():
    numeric_candidates = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    if numeric_candidates:
        selected_record_set_id = rs_id
        numeric_field_id = numeric_candidates[0]  # pick first numeric column
        # try to pick a categorical grouping field, if available
        potential_group_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field_id = potential_group_fields[0] if potential_group_fields else None
        break

if not selected_record_set_id or not numeric_field_id:
    print("No record set with numeric fields found.")
else:
    print(f"Using record set: {selected_record_set_id}")
    print(f"Using numeric field (by @id): {numeric_field_id}")
    if group_field_id:
        print(f"Using group field (by @id): {group_field_id}")
    print()
    df = dataframes[selected_record_set_id]
    # Filter
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std())
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())


## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized form.
You can also plot the means by groups (if a group field is available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    df = dataframes[selected_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} (by @id)")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # Normalized boxplot for filtered data
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=filtered_df[norm_col])
        plt.title(f"Boxplot of normalized {numeric_field_id} (> threshold)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.show()

    # Grouped barplot
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion
This notebook demonstrates how to access and analyze the FAIR² dataset using the `mlcroissant` library. By referencing all entities (record sets and fields) by their `@id`, we ensure reproducibility and adherence to the Croissant standard. Typical EDA steps—filtering by threshold, normalization, grouping, and visualization—can reveal variable distributions and differences across demographic or other groupings. Apply similar steps to deeper analyses, model building, or policy-exploration tasks in reproducible fashion.